In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Get started with Gemini-TTS Voice Replication

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/get_started_with_gemini_tts_vr_eap.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Faudio%2Fspeech%2Fgetting-started%2Fget_started_with_gemini_tts_vr_eap.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/audio/speech/getting-started/get_started_with_gemini_tts_vr_eap.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/get_started_with_gemini_tts_vr_eap.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/get_started_with_gemini_tts_vr_eap.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/get_started_with_gemini_tts_vr_eap.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/get_started_with_gemini_tts_vr_eap.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/get_started_with_gemini_tts_vr_eap.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/audio/speech/getting-started/get_started_with_gemini_tts_vr_eap.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>



This is an Early Access Program demo of Voice Replication feature on Gemini 3 TTS models. Access is limited.

### Prerequisites:

1. Use the default [Python 3 Runtime](https://screenshot.googleplex.com/A7A6tR4NoFWMvAd) to run this colab.

2. **in case of permission failures:** Assign `roles/aiplatform.user` role for your authenticated user in the selected Google Cloud project above.

3. **in case of API not enabled errors:** Enable Agent Platform API (aiplatform.googleapis.com) for your project. [See instructions](https://docs.cloud.google.com/endpoints/docs/openapi/enable-api)

### To use this colab:

1. Run the `authenticate` block. This is where you enter your PROJECT_ID.
2. Run `Voice Replication UI` block: This code will display UI where you can record your voice with your default microphone (built-in computer microphone).
3. **Microphone permission is required**. The interaction to allow microphone access can be awkward and confusing. You should first see a microphone access pop up. After accepting it, rerun the cell. This time the browser should prompt for microphone access. Make sure to check the url bar to confirm you have given the microphone access.


In [ ]:
# @title authenticate
import sys
import os

# fmt: off
PROJECT_ID = ""  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

LOCATION = "global"

! gcloud config set project {PROJECT_ID}
! gcloud auth application-default login -q
! gcloud auth application-default set-quota-project {PROJECT_ID}

In [ ]:
#@title Replicate Your Voice
ENVIRONMENT = "PROD"

import base64
import os
import subprocess
import tempfile
import requests
import json
import wave
import io
from google.colab import output
from IPython.display import HTML, display

def convert_to_24k_wav(audio_bytes):
    """Converts raw audio bytes (WebM/Ogg) to 24kHz WAV and removes silence using ffmpeg."""
    with tempfile.NamedTemporaryFile(delete=False, suffix='.webm') as temp_in:
        temp_in.write(audio_bytes)
        temp_in_path = temp_in.name

    temp_processed_path = temp_in_path + '_processed.wav'

    # Use ffmpeg to convert to 24000 Hz, mono, wav format and remove silence
    # silenceremove=1:0:0.01:-30dB:1  (start_trim: detection_duration: threshold_duration: threshold_level: end_trim)
    # The 'start_trim=1' means it will remove silence at the start.
    # The 'end_trim=1' means it will remove silence at the end.
    # '0:0.01:-30dB' means silence is detected if audio is below -30dB for at least 0.01 seconds.
    subprocess.run([
        'ffmpeg', '-y', '-i', temp_in_path,
        '-ar', '24000', '-ac', '1', '-f', 'wav', temp_processed_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    with open(temp_processed_path, 'rb') as f_out:
        wav_bytes = f_out.read()

    os.remove(temp_in_path)
    os.remove(temp_processed_path)

    return wav_bytes

# New function to convert webm to wav for download
def convert_webm_to_wav_for_download(webm_b64):
    try:
        webm_bytes = base64.b64decode(webm_b64)
        wav_bytes = convert_to_24k_wav(webm_bytes)
        return base64.b64encode(wav_bytes).decode('utf-8')
    except Exception as e:
        return f"Error converting webm to wav: {str(e)}"

def create_replicated_voice_key(model_name, source_audio_b64, consent_audio_b64, project_id, access_token, environment):
    try:
        base_url = "https://aiplatform.googleapis.com"

        url_voices = f"{base_url}/v1beta1/projects/{project_id}/locations/global/voices"

        payload = {
            "voice": {
                "model": f"models/{model_name}",
                "type": "REPLICATED",
                "replicated": {
                    "source_audio": {
                        "mime_type": "audio/wav;rate=24000",
                        "data": source_audio_b64
                    },
                    "consent_audio": {
                        "mime_type": "audio/wav;rate=24000",
                        "data": consent_audio_b64
                    }
                }
            },
            "store": False
        }

        headers = {
            "Authorization": f"Bearer {access_token}",
            "x-goog-user-project": project_id,
            "Content-Type": "application/json; charset=utf-8"
        }

        response = requests.post(url_voices, headers=headers, json=payload)

        if response.status_code == 200:
            resp_json = response.json()
            voice_key = resp_json.get("key")
            if voice_key:
                return {"success": True, "key": voice_key}
            else:
                return {"success": False, "error": f"Key not found in response. Raw Response: {response.text[:200]}"}
        else:
            return {"success": False, "error": f"Voice creation API Error: {response.status_code} - {response.text[:300]}"}

    except Exception as e:
        return {"success": False, "error": f"Voice creation Python Error: {str(e)}"}

# Wrapper function to be called from JavaScript for key generation
def generate_voice_key_rpc(source_b64, consent_b64, model_id_for_key):
    try:
        access_token = subprocess.getoutput("gcloud auth print-access-token").strip()
        project_id = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)

        # First convert source and consent audio to WAV (if WebM) and ensure 24kHz
        source_audio_bytes = base64.b64decode(source_b64)
        consent_audio_bytes = base64.b64decode(consent_b64)

        is_source_webm = source_audio_bytes.startswith(b'\x1A\x45\xDF\xA3')
        is_consent_webm = consent_audio_bytes.startswith(b'\x1A\x45\xDF\xA3')

        if is_source_webm:
            source_wav = convert_to_24k_wav(source_audio_bytes)
        else:
            with tempfile.NamedTemporaryFile(delete=False, suffix='.wav') as temp_in:
                temp_in.write(source_audio_bytes)
                temp_in_path = temp_in.name
            temp_processed_path = temp_in_path + '_processed.wav'
            subprocess.run([
                'ffmpeg', '-y', '-i', temp_in_path,
                '-ar', '24000', '-ac', '1', '-f', 'wav', temp_processed_path
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            with open(temp_processed_path, 'rb') as f_out:
                source_wav = f_out.read()
            os.remove(temp_in_path)
            os.remove(temp_processed_path)

        if is_consent_webm:
            consent_wav = convert_to_24k_wav(consent_audio_bytes)
        else:
            with tempfile.NamedTemporaryFile(delete=False, suffix='.wav') as temp_in:
                temp_in.write(consent_audio_bytes)
                temp_in_path = temp_in.name
            temp_processed_path = temp_in_path + '_processed.wav'
            subprocess.run([
                'ffmpeg', '-y', '-i', temp_in_path,
                '-ar', '24000', '-ac', '1', '-f', 'wav', temp_processed_path
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            with open(temp_processed_path, 'rb') as f_out:
                consent_wav = f_out.read()
            os.remove(temp_in_path)
            os.remove(temp_processed_path)

        final_source_b64 = base64.b64encode(source_wav).decode('utf-8')
        final_consent_b64 = base64.b64encode(consent_wav).decode('utf-8')

        result = create_replicated_voice_key(model_id_for_key, final_source_b64, final_consent_b64, project_id, access_token, ENVIRONMENT)
        if result["success"]:
            return f"SUCCESS_KEY:{result['key']}"
        else:
            return f"ERROR_KEY:{result['error']}"
    except Exception as e:
        return f"ERROR_KEY:Python Error during key generation: {str(e)}"


# 1. Define the Python function that will receive the audio and make the RPC call
def process_audio_rpc(replicated_voice_key, text_to_synthesize):
    try:
        # ---------------------------------------------------------
        # Agent Platform REST API Call
        # ---------------------------------------------------------
        # Get auth token and project details
        access_token = subprocess.getoutput("gcloud auth print-access-token").strip()
        project_id = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)

        model_id = "gemini-3.1-flash-tts-preview"
        base_url = "https://aiplatform.googleapis.com"
        if ENVIRONMENT == "STAGING":
            base_url = "https://staging-aiplatform.sandbox.googleapis.com"
        elif ENVIRONMENT == "AUTOPUSH":
            base_url = "https://autopush-aiplatform.sandbox.googleapis.com"

        url_vertex = f"{base_url}/v1/projects/{project_id}/locations/global/publishers/google/models/{model_id}:generateContent"

        payload = {
            "contents": [{"role": "user", "parts": [{"text": text_to_synthesize}]}],
            "generationConfig": {
                "responseModalities": ["AUDIO"],
                "speechConfig": {
                    "voiceConfig": {
                      "voice": replicated_voice_key # Use the key here
                    }
                }
            }
        }

        headers = {
            "Authorization": f"Bearer {access_token}",
            "x-goog-user-project": project_id,
            "Content-Type": "application/json; charset=utf-8"
        }

        response = requests.post(url_vertex, headers=headers, json=payload)

        if response.status_code == 200:
            try:
                resp_json = response.json()
                b64_out = resp_json["candidates"][0]["content"]["parts"][0]["inlineData"]["data"]

                raw_audio = base64.b64decode(b64_out)
                # Check if it already has RIFF header (starts with b'RIFF')
                if not raw_audio.startswith(b'RIFF'):
                    # Add WAV header for 24kHz 16-bit mono PCM
                    wav_io = io.BytesIO()
                    with wave.open(wav_io, 'wb') as wav_file:
                        wav_file.setnchannels(1)
                        wav_file.setsampwidth(2) # 16-bit
                        wav_file.setframerate(24000)
                        wav_file.writeframes(raw_audio)
                    b64_out = base64.b64encode(wav_io.getvalue()).decode('utf-8')

                return f"SUCCESS_AUDIO:{b64_out}"
            except Exception as e:
                return f"Parse Error: {str(e)} - Raw Response: {response.text[:200]}"
        else:
            return f"API Error: {response.status_code} - {response.text[:300]}"

    except Exception as e:
        return f"Python Error: {str(e)}"

# 2. Register this function so the JavaScript frontend can see and call it
output.register_callback('process_audio_rpc', process_audio_rpc)
output.register_callback('convert_webm_to_wav', convert_webm_to_wav_for_download)
output.register_callback('generate_voice_key_rpc', generate_voice_key_rpc)

# 3. Define the HTML and JavaScript UI as a raw string to prevent Python from parsing JS escape characters
html_ui = r"""
<div style="font-family: sans-serif; padding: 20px; border: 1px solid #ccc; border-radius: 8px; max-width: 600px;">
    <h2 style="margin-top: 0;">Voice Replication TTS</h2>

    <!-- Section 1: Record Source Audio -->
    <div style="margin-bottom: 20px; padding: 15px; border: 1px solid #eee; border-radius: 4px;">
        <h3>1. Record Source Audio</h3>
        <p style="font-size: 14px; color: #555;">
            Record up to 30 seconds of your voice. Need inspiration? Read this:<br>
            <i id="inspirationQuote"></i>
        </p>
        <div style="display: flex; align-items: center; gap: 10px;">
            <button id="btnSource" style="padding: 8px 12px; background-color: #4CAF50; color: white; border: none; border-radius: 4px; cursor: pointer;">Start Recording</button>
            <button id="btnSourceUpload" style="padding: 8px 12px; background-color: #6C757D; color: white; border: none; border-radius: 4px; cursor: pointer;">Upload WAV</button>
            <a id="btnSourceDownload" href="#" download="source_audio.wav" style="padding: 8px 12px; background-color: #007BFF; color: white; border: none; border-radius: 4px; cursor: pointer; text-decoration: none; opacity: 0.5; pointer-events: none;">Download WAV</a>
        </div>
        <div id="audioSourceContainer" style="margin-top: 10px;"></div>
    </div>

    <!-- Section 2: Record Consent Audio -->
    <div style="margin-bottom: 20px; padding: 15px; border: 1px solid #eee; border-radius: 4px;">
        <h3>2. Record Consent Audio</h3>
        <p style="font-size: 14px; color: #555;">
            Please say exactly:<br>
            <i>"I am the owner of this voice and have consented to the creation of a synthetic model of my voice through the use of Google Cloud."</i>
        </p>
        <div style="display: flex; align-items: center; gap: 10px;">
            <button id="btnConsent" style="padding: 8px 12px; background-color: #4CAF50; color: white; border: none; border-radius: 4px; cursor: pointer;">Start Recording</button>
            <button id="btnConsentUpload" style="padding: 8px 12px; background-color: #6C757D; color: white; border: none; border-radius: 4px; cursor: pointer;">Upload WAV</button>
            <a id="btnConsentDownload" href="#" download="consent_audio.wav" style="padding: 8px 12px; background-color: #007BFF; color: white; border: none; border-radius: 4px; cursor: pointer; text-decoration: none; opacity: 0.5; pointer-events: none;">Download WAV</a>
        </div>
        <div id="audioConsentContainer" style="margin-top: 10px;"></div>
    </div>

    <!-- Section 3: Voice Key -->
    <div style="margin-bottom: 20px; padding: 15px; border: 1px solid #eee; border-radius: 4px;">
        <h3>3. Voice Key</h3>
        <textarea id="voiceKeyTextarea" rows="4" style="width: 100%; box-sizing: border-box; padding: 8px; font-family: monospace;" placeholder="Voice key will appear here or can be uploaded..."></textarea>
        <div style="display: flex; align-items: center; gap: 10px; margin-top: 10px;">
            <button id="btnGenerateKey" style="padding: 8px 12px; background-color: #FFC107; color: black; border: none; border-radius: 4px; cursor: pointer;" disabled>Generate Key</button>
            <button id="btnUploadKey" style="padding: 8px 12px; background-color: #17A2B8; color: white; border: none; border-radius: 4px; cursor: pointer;">Upload Key</button>
            <a id="btnKeyDownload" href="#" download="voice_key.txt" style="padding: 8px 12px; background-color: #007BFF; color: white; border: none; border-radius: 4px; cursor: pointer; text-decoration: none; opacity: 0.5; pointer-events: none;">Download Key</a>
        </div>
    </div>

    <!-- Section 4: Text to Synthesize -->
    <div style="margin-bottom: 20px; padding: 15px; border: 1px solid #eee; border-radius: 4px;">
        <h3>4. Text to Synthesize</h3>
        <textarea id="ttsText" rows="6" style="width: 100%; box-sizing: border-box; padding: 8px;" placeholder="Enter text to synthesize...">Read out the following with a natural, human-like voice:

Quantum computing leverages the principles of superposition and entanglement to perform complex calculations far beyond the reach of classical computers, promising a revolution in technology and science.</textarea>
        <div style="margin-top: 15px;">
            <button id="btnSynthesize" style="padding: 12px 20px; background-color: #2196F3; color: white; border: none; border-radius: 4px; cursor: pointer; font-size: 16px;" disabled>Synthesize Speech</button>
            <div id="resultAudioContainer" style="margin-top: 15px;"></div>
        </div>
    </div>

</div>
    <p id="statusText" style="margin-top: 15px; font-weight: bold; color: #333;">Status: Ready.</p>

<script>
    // List of longer inspirational quotes (around 15-20 seconds to read)
    const quotes = [
        '"It is not the critic who counts; not the man who points out how the strong man stumbles, or where the doer of deeds could have done them better. The credit belongs to the man who is actually in the arena, whose face is marred by dust and sweat and blood; who strives valiantly..." - Theodore Roosevelt',
        '"Here\'s to the crazy ones. The misfits. The rebels. The troublemakers. The round pegs in the square holes. The ones who see things differently. They\'re not fond of rules. And they have no respect for the status quo. You can quote them, disagree with them, glorify or vilify them. About the only thing you can\'t do is ignore them." - Steve Jobs',
        '"Our deepest fear is not that we are inadequate. Our deepest fear is that we are powerful beyond measure. It is our light, not our darkness that most frightens us. We ask ourselves, \'Who am I to be brilliant, gorgeous, talented, fabulous?\' Actually, who are you not to be?" - Marianne Williamson',
        '"The master in the art of living makes little distinction between his work and his play, his labor and his leisure, his mind and his body, his information and his recreation, his love and his religion. He hardly knows which is which. He simply pursues his vision of excellence at whatever he does, leaving others to decide whether he is working or playing." - L.P. Jacks',
        '"I wanted a perfect ending. Now I\'ve learned, the hard way, that some poems don\'t rhyme, and some stories don\'t have a clear beginning, middle, and end. Life is about not knowing, having to change, taking the moment and making the best of it, without knowing what\'s going to happen next. Delicious Ambiguity." - Gilda Radner'
    ];

    // Select a random quote and display it
    const randomQuote = quotes[Math.floor(Math.random() * quotes.length)];
    document.getElementById('inspirationQuote').innerText = randomQuote;

    let sourceB64 = null;
    let consentB64 = null;
    let voiceKey = null; // New variable for the voice key

    const statusText = document.getElementById('statusText'); // Make statusText globally accessible for updates
    const voiceKeyTextarea = document.getElementById('voiceKeyTextarea');
    const btnGenerateKey = document.getElementById('btnGenerateKey');
    const btnUploadKey = document.getElementById('btnUploadKey');
    const btnKeyDownload = document.getElementById('btnKeyDownload'); // New key download button
    const btnSynthesize = document.getElementById('btnSynthesize');

    // Function to update the state of the Generate Key button
    function updateGenerateKeyButtonState() {
        if (sourceB64 && consentB64) {
            btnGenerateKey.disabled = false;
            btnGenerateKey.style.opacity = '1';
        } else {
            btnGenerateKey.disabled = true;
            btnGenerateKey.style.opacity = '0.5';
        }
    }

    // Function to update the state of the Synthesize button
    function updateSynthesizeButtonState() {
        if (voiceKey && ttsText.value.trim()) {
            btnSynthesize.disabled = false;
            btnSynthesize.style.opacity = '1';
        } else {
            btnSynthesize.disabled = true;
            btnSynthesize.style.opacity = '0.5';
        }
    }

    // Function to update the state of the Key Download button
    function updateKeyDownloadButtonState() {
        if (voiceKeyTextarea.value.trim() && voiceKey) { // Ensure voiceKey is also set
            btnKeyDownload.disabled = false;
            btnKeyDownload.style.opacity = '1';
            btnKeyDownload.style.pointerEvents = 'auto';
            const blob = new Blob([voiceKeyTextarea.value], { type: 'text/plain' });
            btnKeyDownload.href = URL.createObjectURL(blob);
        } else {
            btnKeyDownload.disabled = true;
            btnKeyDownload.style.opacity = '0.5';
            btnKeyDownload.style.pointerEvents = 'none';
            btnKeyDownload.href = '#';
        }
    }

    // Call this initially and whenever sourceB64 or consentB64 changes
    updateGenerateKeyButtonState();
    updateSynthesizeButtonState(); // Initial call for synthesize button
    updateKeyDownloadButtonState(); // Initial call for key download button

    // Setup recorders, now calling updateGenerateKeyButtonState on audio ready
    setupRecorder('btnSource', 'btnSourceUpload', 'btnSourceDownload', 'audioSourceContainer', b64 => { sourceB64 = b64; updateGenerateKeyButtonState(); }, 'source_audio');
    setupRecorder('btnConsent', 'btnConsentUpload', 'btnConsentDownload', 'audioConsentContainer', b64 => { consentB64 = b64; updateGenerateKeyButtonState(); }, 'consent_audio');

    // Add event listener for Generate Key button
    btnGenerateKey.addEventListener('click', async () => {
        if (!sourceB64 || !consentB64) {
            statusText.innerText = "Error: Both Source and Consent Audio must be recorded or uploaded to generate a key.";
            statusText.style.color = "red";
            return;
        }

        statusText.style.color = "#333";
        statusText.innerText = "Status: Generating voice key...";
        voiceKeyTextarea.value = "Generating key...";
        voiceKey = null;
        updateSynthesizeButtonState(); // Disable synthesize button while key is being generated
        updateKeyDownloadButtonState(); // Disable download button while key is being generated

        const modelIdForKey = "gemini-3.1-flash-tts-preview"; // Hardcoded model for key generation

        try {
            const keyResult = await google.colab.kernel.invokeFunction('generate_voice_key_rpc', [sourceB64, consentB64, modelIdForKey], {});
            let pyResponse = keyResult.data['text/plain'];
            pyResponse = pyResponse.replace(/^['"](.*)['"]$/, '$1').replace(/\n/g, '\n');

            if (pyResponse.startsWith("SUCCESS_KEY:")) {
                voiceKey = pyResponse.substring("SUCCESS_KEY:".length);
                voiceKeyTextarea.value = voiceKey;
                statusText.innerText = "Status: Voice key generated successfully!";
            } else {
                voiceKeyTextarea.value = "Error generating key.";
                statusText.innerText = "Result: " + pyResponse;
                voiceKey = null;
            }
        } catch (err) {
            voiceKeyTextarea.value = "Python RPC error during key generation.";
            statusText.innerText = "Error calling Python for key generation: " + err;
            voiceKey = null;
        } finally {
            updateSynthesizeButtonState(); // Re-enable or disable synthesize button based on new voiceKey state
            updateKeyDownloadButtonState(); // Update key download button state based on new voiceKey state
        }
    });

    // Add event listener for Upload Key button
    btnUploadKey.addEventListener('click', () => {
        const fileInput = document.createElement('input');
        fileInput.type = 'file';
        fileInput.accept = 'text/plain'; // Assuming key is in a text file
        fileInput.style.display = 'none';
        document.body.appendChild(fileInput);

        fileInput.addEventListener('change', (event) => {
            const file = event.target.files[0];
            if (file) {
                const reader = new FileReader();
                reader.readAsText(file);
                reader.onloadend = () => {
                    voiceKey = reader.result.trim();
                    voiceKeyTextarea.value = voiceKey;
                    statusText.innerText = "Status: Voice key uploaded successfully.";
                    updateSynthesizeButtonState(); // Update synthesize button state after upload
                    updateKeyDownloadButtonState(); // Update key download button state after upload
                };
            } else {
                statusText.innerText = "Status: No file selected.";
            }
            document.body.removeChild(fileInput); // Clean up the hidden input
        });
        fileInput.click();
    });

    // Main Synthesize Logic
    const ttsText = document.getElementById('ttsText');
    const resultContainer = document.getElementById('resultAudioContainer');

    // Event listener for changes in the text area to update button state
    ttsText.addEventListener('input', updateSynthesizeButtonState);

    // Event listener for changes in voiceKeyTextarea to update key download button state
    voiceKeyTextarea.addEventListener('input', updateKeyDownloadButtonState);

    btnSynthesize.addEventListener('click', () => {
        if (!voiceKey) {
            statusText.innerText = "Error: Please generate or upload a Voice Key first.";
            statusText.style.color = "red";
            return;
        }
        if (!ttsText.value.trim()) {
            statusText.innerText = "Error: Please enter the text you want to synthesize.";
            statusText.style.color = "red";
            return;
        }

        statusText.style.color = "#333";
        statusText.innerText = "Status: Calling Agent Platform for synthesis...";
        resultContainer.innerHTML = ''; // Clear previous result audio

        // Pass all user input to the Python RPC function
        google.colab.kernel.invokeFunction('process_audio_rpc', [voiceKey, ttsText.value], {})
            .then(result => {
                let pyResponse = result.data['text/plain'];
                // Remove surrounding quotes from colab output
                pyResponse = pyResponse.replace(/^['"](.*)['"]$/, '$1').replace(/\n/g, '\n');

                if (pyResponse.startsWith("SUCCESS_AUDIO:")) {
                    const b64Audio = pyResponse.substring("SUCCESS_AUDIO:".length);
                    statusText.innerText = "Status: Synthesis successful!";

                    const audioElement = document.createElement('audio');
                    audioElement.controls = true;
                    audioElement.src = "data:audio/wav;base64," + b64Audio;
                    resultContainer.appendChild(audioElement);
                } else {
                    statusText.innerText = "Result: " + pyResponse;
                }
            }).catch(err => {
                statusText.innerText = "Error calling Python: " + err;
            });
    });


    function setupRecorder(btnRecordId, btnUploadId, downloadBtnId, containerId, onAudioReady, fileNamePrefix) {
        let isRecording = false;
        let mediaRecorder;
        let audioChunks = [];
        let timeoutId = null;
        const btnRecord = document.getElementById(btnRecordId);
        const btnUpload = document.getElementById(btnUploadId);
        const fileInput = document.createElement('input'); // Hidden file input
        fileInput.type = 'file';
        fileInput.accept = 'audio/wav';
        fileInput.style.display = 'none';
        document.body.appendChild(fileInput); // Append to body to avoid layout issues

        const container = document.getElementById(containerId);
        const downloadBtn = document.getElementById(downloadBtnId);

        // Function to reset UI state for this section
        function resetUI() {
            container.innerHTML = '';
            downloadBtn.disabled = true;
            downloadBtn.style.opacity = '0.5';
            downloadBtn.style.pointerEvents = 'none';
            downloadBtn.href = '#';
            btnRecord.disabled = false;
            btnRecord.style.opacity = '1';
            btnRecord.innerText = "Start Recording";
            btnRecord.style.backgroundColor = "#4CAF50";
            btnUpload.disabled = false;
            btnUpload.style.opacity = '1';
            onAudioReady(null); // Clear the stored B64 when resetting
        }

        btnRecord.addEventListener('click', async () => {
            if (!isRecording) {
                resetUI(); // Reset UI before starting new action
                btnUpload.disabled = true; // Disable upload when recording
                btnUpload.style.opacity = '0.5';

                try {
                    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
                    mediaRecorder = new MediaRecorder(stream);
                    audioChunks = [];

                    mediaRecorder.ondataavailable = event => {
                        audioChunks.push(event.data);
                    };

                    mediaRecorder.onstop = () => {
                        clearTimeout(timeoutId);
                        const audioBlob = new Blob(audioChunks, { type: 'audio/webm' });

                        const audioUrl = URL.createObjectURL(audioBlob);
                        const audioElement = document.createElement('audio');
                        audioElement.controls = true;
                        audioElement.src = audioUrl;
                        container.appendChild(audioElement);

                        const reader = new FileReader();
                        reader.readAsDataURL(audioBlob);
                        reader.onloadend = async () => {
                            const webmB64 = reader.result.split(',')[1];
                            onAudioReady(webmB64); // This updates sourceB64/consentB64 with WebM

                            statusText.innerText = "Status: Converting recorded audio to WAV...";
                            try {
                                const wavB64Result = await google.colab.kernel.invokeFunction('convert_webm_to_wav', [webmB64], {});
                                let wavB64 = wavB64Result.data['text/plain'];
                                wavB64 = wavB64.replace(/^['"](.*)['"]$/, '$1').replace(/\n/g, '\n'); // Clean up output

                                if (wavB64.startsWith("Error")) {
                                    statusText.innerText = "Error during WAV conversion: " + wavB64;
                                } else {
                                    downloadBtn.href = "data:audio/wav;base64," + wavB64;
                                    downloadBtn.download = `${fileNamePrefix}_${new Date().toISOString().slice(0,10)}.wav`;
                                    downloadBtn.disabled = false;
                                    downloadBtn.style.opacity = '1';
                                    downloadBtn.style.pointerEvents = 'auto';
                                    statusText.innerText = "Status: Audio recorded and ready for download.";
                                }
                            } catch (error) {
                                statusText.innerText = "Error invoking WAV conversion: " + error;
                            }
                        };

                        stream.getTracks().forEach(track => track.stop());
                    };

                    mediaRecorder.start();
                    isRecording = true;
                    btnRecord.innerText = "Stop Recording";
                    btnRecord.style.backgroundColor = "#f44336";

                    timeoutId = setTimeout(() => {
                        if (mediaRecorder.state !== 'inactive') {
                            mediaRecorder.stop();
                            isRecording = false;
                            btnRecord.innerText = "Start Recording";
                            btnRecord.style.backgroundColor = "#4CAF50";
                        }
                    }, 30000);

                } catch (err) {
                    container.innerHTML = `<p style="color: red; font-size: 14px;">Microphone error: ${err.message}</p>`;
                    resetUI();
                }
            } else {
                if (mediaRecorder && mediaRecorder.state !== 'inactive') {
                    mediaRecorder.stop();
                }
                isRecording = false;
                btnRecord.innerText = "Start Recording";
                btnRecord.style.backgroundColor = "#4CAF50";
            }
        });

        // Upload button event listener
        btnUpload.addEventListener('click', () => {
            resetUI(); // Reset UI before starting new action
            btnRecord.disabled = true; // Disable record when uploading
            btnRecord.style.opacity = '0.5';
            fileInput.click(); // Trigger the hidden file input
        });

        fileInput.addEventListener('change', (event) => {
            const file = event.target.files[0];
            if (file) {
                if (file.type !== 'audio/wav') {
                    statusText.innerText = "Error: Please upload a WAV file.";
                    statusText.style.color = "red";
                    resetUI();
                    return;
                }

                const reader = new FileReader();
                reader.readAsDataURL(file);
                reader.onloadend = () => {
                    const wavB64 = reader.result.split(',')[1];
                    onAudioReady(wavB64); // Store the WAV directly

                    const audioUrl = URL.createObjectURL(file);
                    const audioElement = document.createElement('audio');
                    audioElement.controls = true;
                    audioElement.src = audioUrl;
                    container.appendChild(audioElement);

                    downloadBtn.href = "data:audio/wav;base64," + wavB64;
                    downloadBtn.download = `${fileNamePrefix}_${file.name}`;
                    downloadBtn.disabled = false;
                    downloadBtn.style.opacity = '1';
                    downloadBtn.style.pointerEvents = 'auto';
                    statusText.innerText = "Status: WAV file uploaded and ready for synthesis/download.";
                };
            } else {
                resetUI(); // If no file selected, reset UI
            }
        });
    }

</script>
"""

# 4. Render the UI in the cell output
display(HTML(html_ui))